In [0]:
dbutils.widgets.text("catalog_name","dbr_dev_ua_5816_trail_2")
dbutils.widgets.text("gold_schema","viktoriia_kalenichenko_gold")

In [0]:
CATALOG = dbutils.widgets.get("catalog_name")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

DEMO = f"{GOLD}.governance_demo"
ROLES = f"{GOLD}.access_roles"
RLS_FUNCTION = f"{GOLD}.rls_subregion_filter"

Gold schema: dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold
Demo table: dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.governance_demo
Roles table: dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.access_roles
RLS function: dbr_dev_ua_5816_trail_2.viktoriia_kalenichenko_gold.rls_subregion_filter


In [0]:
spark.sql(f"""
    ALTER TABLE {DEMO}
    SET ROW FILTER {RLS_FUNCTION} ON (subregion)
""")

DataFrame[]

In [0]:
# Check current user and assigned role
print("Current user:")
spark.sql("SELECT current_user() AS user_email").show(truncate=False)

print("Assigned role:")
spark.sql(f"""
    SELECT user_email, role, allowed_subregion
    FROM {ROLES}
    WHERE user_email = current_user()
""").show(truncate=False)

# Check rows visible after applying RLS
print("Visible rows:")
spark.sql(f"""
    SELECT
        COUNT(*) AS visible_rows,
        COUNT(DISTINCT country_name) AS visible_countries
    FROM {DEMO}
""").show()

Current user:
+--------------------------------+
|user_email                      |
+--------------------------------+
|victoriya44250@softserve.academy|
+--------------------------------+

Assigned role:
+--------------------------------+-------+-----------------+
|user_email                      |role   |allowed_subregion|
+--------------------------------+-------+-----------------+
|victoriya44250@softserve.academy|analyst|NULL             |
+--------------------------------+-------+-----------------+

Visible rows:
+------------+-----------------+
|visible_rows|visible_countries|
+------------+-----------------+
|        1146|               38|
+------------+-----------------+



In [0]:
CLS_FUNCTION = f"{GOLD}.mask_production"

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CLS_FUNCTION}(
    production_value DOUBLE
)
RETURNS DOUBLE
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {ROLES}
        WHERE user_email = current_user()
          AND role = 'admin'
    )
    THEN production_value
    ELSE CAST(NULL AS DOUBLE)
END
""")


DataFrame[]

In [0]:
spark.sql(f"""
    ALTER TABLE {DEMO}
    ALTER COLUMN total_production
    SET MASK {CLS_FUNCTION}
""")

Column-Level Security applied successfully!


In [0]:
display(
    spark.sql(f"""
        SELECT
            country_name,
            subregion,
            year,
            total_production
        FROM {DEMO}
        ORDER BY country_name, year
        LIMIT 10
    """)
)

country_name,subregion,year,total_production
Albania,Southern Europe,1990,null
Albania,Southern Europe,1991,null
Albania,Southern Europe,1992,null
Albania,Southern Europe,1993,null
Albania,Southern Europe,1994,null
Albania,Southern Europe,1995,null
Albania,Southern Europe,1996,null
Albania,Southern Europe,1997,null
Albania,Southern Europe,1998,null
Albania,Southern Europe,1999,null
